Classify Text Sentiment — Preprocess the text (lowercase, remove punctuation/stopwords, tokenize, stem or lemmatize), extract features using Bag-of-Words or TF-IDF, then train a text classification model (e.g. Naive Bayes or Logistic Regression) to predict the label. Report Accuracy and F1-score, and show the top 10 most important words per class. https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews

Import libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

Download NLTK resources

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

Load dataset

In [3]:
df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")

df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


Check the columns

In [4]:
print(df.shape)
print(df.columns)

(23486, 11)
Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')


Keep the required columns

In [5]:
df = df[['Review Text', 'Rating']]

df = df.dropna()

df.head()

,Review Text,Rating
0,Absolutely wonderful - silky and sexy and comf...,4
1,Love this dress! it's sooo pretty. i happene...,5
2,I had such high hopes for this dress and reall...,3
3,"I love, love, love this jumpsuit. it's fun, fl...",5
4,This shirt is very flattering to all due to th...,5


Create sentiment labels

In [6]:
def create_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

df["Sentiment"] = df["Rating"].apply(create_sentiment)

df["Sentiment"].value_counts()

,count
Sentiment,
Positive,17448
Neutral,2823
Negative,2370


Preprocess the text

In [7]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    # lowercase
    text = text.lower()

    # remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # tokenize
    words = text.split()

    # remove stopwords
    words = [word for word in words if word not in stop_words]

    # lemmatize
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [8]:
df["Clean_Text"] = df["Review Text"].apply(preprocess)

df[["Review Text", "Clean_Text"]].head()

,Review Text,Clean_Text
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky sexy comfortable
1,Love this dress! it's sooo pretty. i happene...,love dress sooo pretty happened find store im ...
2,I had such high hopes for this dress and reall...,high hope dress really wanted work initially o...
3,"I love, love, love this jumpsuit. it's fun, fl...",love love love jumpsuit fun flirty fabulous ev...
4,This shirt is very flattering to all due to th...,shirt flattering due adjustable front tie perf...


Separate X and y

In [9]:
X = df["Clean_Text"]
y = df["Sentiment"]

Train-test split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Convert text to TF-IDF features

In [11]:
tfidf = TfidfVectorizer(
    max_features=5000
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

Train Logistic Regression

In [12]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

Make predictions

In [13]:
y_pred = model.predict(X_test_tfidf)

Accuracy and F1-score

In [14]:
accuracy = accuracy_score(y_test, y_pred)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

Accuracy: 0.7646279531905498
F1 Score: 0.7875153399912072


Classification report

In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Negative       0.45      0.61      0.52       474
     Neutral       0.32      0.49      0.39       565
    Positive       0.96      0.83      0.89      3490

    accuracy                           0.76      4529
   macro avg       0.58      0.64      0.60      4529
weighted avg       0.82      0.76      0.79      4529



Top 10 important words per class

In [16]:
feature_names = np.array(tfidf.get_feature_names_out())

for i, class_name in enumerate(model.classes_):

    top10 = np.argsort(model.coef_[i])[-10:]

    print("\n", class_name)
    print(feature_names[top10])


 Negative
['asap' 'bummed' 'worst' 'unflattering' 'wanted' 'poor' 'cheap' 'awful'
 'horrible' 'disappointed']

 Neutral
['returning' 'breast' 'seem' 'removed' 'within' 'unfortunately' 'ok'
 'added' 'seemed' 'however']

 Positive
['soft' 'highly' 'happy' 'compliment' 'little' 'perfectly' 'great'
 'comfortable' 'perfect' 'love']
